# [1교시]

#### 오답 구조와 프롬프트 설계가 성능에 영향을 미치는지 확인

In [ ]:
import json
import os
from collections import Counter,defaultdict
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
LABELS = ['DEF', 'RIGHT', 'PROC', 'ORG', 'CRIT', 'ETC']
LABEL_DESC = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항',
}

sample_data = [
    {'id': 'D01', 'category': 'DEF', 'text': '이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.'},
    {'id': 'D02', 'category': 'DEF', 'text': '이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.'},
    {'id': 'D03', 'category': 'DEF', 'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.'},
    {'id': 'R01', 'category': 'RIGHT', 'text': '모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 아니한다.'},
    {'id': 'R02', 'category': 'RIGHT', 'text': '사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.'},
    {'id': 'P01', 'category': 'PROC', 'text': '이 법을 위반한 자는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처한다.'},
    {'id': 'P02', 'category': 'PROC', 'text': '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.'},
    {'id': 'O01', 'category': 'ORG', 'text': '분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.'},
    {'id': 'O02', 'category': 'ORG', 'text': '위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.'},
    {'id': 'C01', 'category': 'CRIT', 'text': '후보자는 선거일 현재 25세 이상인 국민이어야 한다.'},
    {'id': 'C02', 'category': 'CRIT', 'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.'},
    {'id': 'E01', 'category': 'ETC', 'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.'},
    {'id': 'E02', 'category': 'ETC', 'text': '이 법 시행 당시 종전의 규정에 따라 한 처분은 이 법에 따른 처분으로 본다.'},
]

sample_data += [
    {'id': 'R03', 'category': 'RIGHT', 'text': '누구든지 정당한 사유 없이 타인의 개인정보를 수집·이용 또는 제공하여서는 아니 된다.'},
    {'id': 'R04', 'category': 'RIGHT', 'text': '사용자는 근로자에게 최저임금 이상을 지급하여야 한다.'},
    {'id': 'R05', 'category': 'RIGHT', 'text': '행정기관은 민원인의 권익을 보호하기 위하여 필요한 조치를 하여야 한다.'},
    {'id': 'R06', 'category': 'RIGHT', 'text': '모든 국민은 건강하고 쾌적한 환경에서 생활할 권리를 가진다.'},
    {'id': 'R07', 'category': 'RIGHT', 'text': '사업주는 산업재해를 예방하기 위하여 안전 및 보건 조치를 이행하여야 한다.'},
    {'id': 'R08', 'category': 'RIGHT', 'text': '누구든지 폭행, 협박 또는 위계로 타인의 업무를 방해하여서는 아니 된다.'},
    {'id': 'R09', 'category': 'RIGHT', 'text': '국가기관은 법령에서 정한 경우를 제외하고는 개인의 자유와 권리를 제한하여서는 아니 된다.'},
    {'id': 'R10', 'category': 'RIGHT', 'text': '이용자는 자신의 개인정보 처리에 관한 사항을 열람하거나 정정을 요구할 수 있다.'},
    {'id': 'R11', 'category': 'RIGHT', 'text': '공무원은 직무상 알게 된 비밀을 누설하여서는 아니 된다.'},
    {'id': 'R12', 'category': 'RIGHT', 'text': '모든 국민은 법률이 정하는 바에 따라 납세의 의무를 진다.'},
    {'id': 'R13', 'category': 'RIGHT', 'text': '판매업자는 소비자에게 상품의 원산지와 가격 정보를 명확히 표시하여야 한다.'},
    {'id': 'R14', 'category': 'RIGHT', 'text': '아동은 어떠한 형태의 학대와 방임으로부터 보호받을 권리를 가진다.'},
    {'id': 'R15', 'category': 'RIGHT', 'text': '정보통신서비스 제공자는 이용자의 동의 없이 개인정보를 제3자에게 제공하여서는 아니 된다.'},
    {'id': 'R16', 'category': 'RIGHT', 'text': '국가와 지방자치단체는 장애인의 이동권 보장을 위하여 필요한 시책을 마련하여야 한다.'},
    {'id': 'R17', 'category': 'RIGHT', 'text': '누구든지 허위 또는 과장된 표시·광고를 하여 소비자를 기만하여서는 아니 된다.'},
    {'id': 'R18', 'category': 'RIGHT', 'text': '피해자는 범죄로 인한 손해의 배상을 청구할 권리를 가진다.'},
    {'id': 'R19', 'category': 'RIGHT', 'text': '사업자는 이용자의 서비스 이용기록을 안전하게 보관하여야 한다.'},
    {'id': 'R20', 'category': 'RIGHT', 'text': '모든 국민은 인간으로서의 존엄과 가치를 가지며 행복을 추구할 권리를 가진다.'},
]

df_all = pd.DataFrame(sample_data)
df = df_all.copy()

print(f'평가 데이터 로드 완료: {len(df)}건')
df.head()

##### 재현 가능한 평가 데이터
- 동일한 데이터와 평가기준

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
SYSTEM_PROMPT = (
    '너는 한국 법률 조항 분류기다. 반드시 다음 6개 코드 중 하나만 선택하라: ' + ', '.join(LABELS) + '. '
    '응답은 반드시 JSON 한 줄로만 반환한다. 형식: {"label":"DEF","reason":"..."}'
)
SYSTEM_PROMPT

In [ ]:
LABEL_GUIDE = '\n'.join([f'- {label}: {description}' for label, description in LABEL_DESC.items()])
def build_prompt(text, mode='zero', examples=None, rules=None, label_only=False):
    base = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하라.',
        '라벨 설명:',
        LABEL_GUIDE,
    ]
    if rules:
        base += ['', '구분 규칙:']
        base += [f'- {rule}' for rule in rules]
    if mode == 'one' and examples:
        example = examples[0]
        base += [
            '',
            '예시 1개:',
            f"문장: {example['text']}",
            '정답(JSON): ' + json.dumps(example['output'], ensure_ascii=False),
        ]
    elif mode == 'few' and examples:
        base += ['', f'예시 {len(examples)}개:']
        for index, example in enumerate(examples, 1):
            base += [
                f'예시 {index}:',
                f"문장: {example['text']}",
                '정답(JSON): ' + json.dumps(example['output'], ensure_ascii=False),
            ]
    base += ['', f'분류할 문장: {text}']
    if label_only:
        base += ['JSON은 label만 포함하라.', '{"label":"DEF|RIGHT|PROC|ORG|CRIT|ETC"}']
    else:
        base += ['JSON으로만 응답하라.', '{"label":"DEF|RIGHT|PROC|ORG|CRIT|ETC","reason":"짧은 근거"}']
    return '\n'.join(base)

In [ ]:
print(build_prompt(df['text'][0]))

# [2교시]

In [ ]:
def load_hf_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype='auto',device_map='auto'
    )
    return model, tokenizer

def extract_json(text):
    start = text.find('{')
    end = text.find('}')
    if start == -1 or end == -1 or end<=start:
        return None
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError:
        return None
def predict(model, tokenizer, prompt, max_new_tokens=128, do_sample=False, temperature=None, top_p=None):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text_input], return_tensors='pt').to(model.device)
    generate_kwargs = {
        'max_new_tokens': max_new_tokens,
        'do_sample': do_sample,
        'pad_token_id': tokenizer.eos_token_id,
    }
    if temperature is not None:
        generate_kwargs['temperature'] = temperature
    if top_p is not None:
        generate_kwargs['top_p'] = top_p
    generated_ids = model.generate(**model_inputs, **generate_kwargs)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    parsed = extract_json(raw_response)
    label = parsed.get('label') if parsed else 'PARSE_FAIL'
    reason = parsed.get('reason') if parsed else raw_response[:160]
    return {
        'raw_response': raw_response,
        'label': label,
        'reason': reason,
    }


def normalize_prediction(label):
    if isinstance(label, str) and label in LABELS:
        return label
    return 'PARSE_FAIL'


print('프롬프트/추론 함수 정의 완료')    

In [ ]:
BASE_ZERO_FEW_EXAMPLES = [
    {'text':'신청인은 처분 통지를 받은 날부터 30일 이내에 이의신철을 할 수 있다', 'output': {'label' :'PROC','reason':'불복 절차를 규정한다' }}
]
def run_eval(model, tokenizer, mode='zero', label_only=False, rules=None,predict_kwargs=None):
    rows = []
    for sample in df.to_dict('records'):
        prompt = build_prompt(
            sample['text'],
            mode=mode,
            examples=BASE_ZERO_FEW_EXAMPLES if mode in ('one', 'few') else None,
            rules=rules,
            label_only=label_only,
        )
        predict_args = predict_kwargs or {}
        result = predict(model, tokenizer, prompt, max_new_tokens=128, **predict_args)
        rows.append({
            'id': sample['id'],
            'true_label': sample['category'],
            'predicted_label': normalize_prediction(result['label']),
            'reason': result['reason'],
            'raw_response': result['raw_response'],
            'correct': normalize_prediction(result['label']) == sample['category'],
            'text': sample['text'],
        })
    return pd.DataFrame(rows)

base_model, base_tokenizer =  load_hf_model(BASE_MODEL)

In [ ]:
base_pred_df = run_eval(base_model,base_tokenizer)
base_pred_df

In [ ]:
def metric_summary(pred_df):
    accuracy = float(pred_df['correct'].mean())
    labels = sorted(df['category'].unique())
    precisions = []
    recalls = []
    f1s = []
    for label in labels:
        tp = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] == label)).sum())
        fp = int(((pred_df['true_label'] != label) & (pred_df['predicted_label'] == label)).sum())
        fn = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] != label)).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    return {
        'accuracy': accuracy,
        'macro_precision': float(np.mean(precisions)),
        'macro_recall': float(np.mean(recalls)),
        'macro_f1': float(np.mean(f1s)),
    }

metric_summary(base_pred_df)

##### 라벨별 혼동 행렬

In [ ]:
label_order = LABELS
label_to_index =  {label:index for index, label in enumerate(label_order)}

def confusion_matrix_from_df(pred_df):
    matrix = np.zeros((len(label_order), len(label_order)), dtype=int)
    for _, row in pred_df.iterrows():
        true_label = row['true_label']
        pred_label = row['predicted_label']
        if true_label in label_to_index and pred_label in label_to_index:
            matrix[label_to_index[true_label], label_to_index[pred_label]] += 1
    return matrix


def per_label_metrics(pred_df):
    rows = []
    for label in label_order:
        tp = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] == label)).sum())
        fp = int(((pred_df['true_label'] != label) & (pred_df['predicted_label'] == label)).sum())
        fn = int(((pred_df['true_label'] == label) & (pred_df['predicted_label'] != label)).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        rows.append({
            'label': label,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': int((pred_df['true_label'] == label).sum()),
        })
    return pd.DataFrame(rows)

confusion = confusion_matrix_from_df(base_pred_df)
label_metrics = per_label_metrics(base_pred_df)

In [ ]:
print('혼동행렬')
pd.DataFrame(confusion,index = label_order,columns=label_order)

In [ ]:
print('라벨별 지표')
print(label_metrics)

#### 오답 유형 묶기
- 라벨형태로 묶어 원인을 분석

In [ ]:
wrong_df = base_pred_df[base_pred_df['correct'] == False].copy()
wrong_df['error_type'] = wrong_df.apply(lambda row : f"{row['true_label']}->{row['predicted_label']}",axis=1)
error_group = wrong_df.groupby('error_type').size().reset_index(name='count').sort_values('count',ascending=False)
print('오답유형')
print(error_group)

#### 프롬프트 민감도
- 같은 모델을 기준으로 zero one few label-only , 규칙추가 버전 차이를 비교
- 변화가 크면 병목은 모델크기나 종류가 아니라... 프롬프트 설계에 있다 증명

In [ ]:
prompt_variants = [
    ('zero', {'mode': 'zero', 'label_only': False, 'rules': None}),
    ('one', {'mode': 'one', 'label_only': False, 'rules': None}),
    ('few', {'mode': 'few', 'label_only': False, 'rules': None}),
    ('label_only', {'mode': 'zero', 'label_only': True, 'rules': None}),
    ('rules_added', {'mode': 'zero', 'label_only': False, 'rules': [
        'DEF는 의무, 금지, 책임처럼 행위 기준을 직접 제시하는 문장이다.',
        'RIGHT는 권리, 청구, 자격처럼 주체의 권한을 보장하는 문장이다.',
        'PROC는 신청, 신고, 기간, 절차처럼 진행 과정을 말한다.',
    ]}),
    # Generation/config variants to test without changing model: sampling and temperature
    ('sample_low_temp', {'mode': 'zero', 'label_only': False, 'rules': None, 'predict_kwargs': {'do_sample': True, 'temperature': 0.2}}),
    ('sample_high_temp', {'mode': 'zero', 'label_only': False, 'rules': None, 'predict_kwargs': {'do_sample': True, 'temperature': 0.8}}),
]

prompt_sensitivity_rows = []
for variant_name, config in prompt_variants:
    pred_variant_df = run_eval(
        base_model,
        base_tokenizer,
        mode=config['mode'],
        label_only=config['label_only'],
        rules=config['rules'],
        predict_kwargs=config.get('predict_kwargs'),
    )
    summary = metric_summary(pred_variant_df)
    prompt_sensitivity_rows.append({'variant': variant_name, **summary})
    summary = metric_summary(base_pred_df)
    prompt_sensitivity_rows.append({'variant': variant_name, **summary})

prompt_sensitivity_df = pd.DataFrame(prompt_sensitivity_rows)
print('프롬프트 민감도 비교')
print(prompt_sensitivity_df)

#### 출력 안정성 확인
- 같은 문장을 여러번 질의했을때 라벨이 바뀌는지 확인
- 출력이 흔들리면 모델이전에 샘플링설정, 출력형식, 파싱규칙

In [ ]:
stability_cases = df.head(5).to_dict('records')
stability_rows = []
for sample in stability_cases:
    labels_seen = []
    for _ in range(3):
        result = predict(base_model, base_tokenizer, build_prompt(sample['text']), max_new_tokens=128, do_sample=False)
        labels_seen.append(normalize_prediction(result['label']))
    stability_rows.append({
        'id': sample['id'],
        'true_label': sample['category'],
        'labels_seen': labels_seen,
        'stable': len(set(labels_seen)) == 1,
    })

stability_df = pd.DataFrame(stability_rows)
print('출력 안정성 샘플')
print(stability_df)

# [3교시]

#### RIGHT 계열에 대해서는 모델이 전혀 인지못함
- RIGHT 예시문등을 보강을 해서 성능개선 작업 : AI 로 생성하거나 해당 라벨에 맞는 실제 문구를 찾아서 셈플링해서 모델을 다시 적용

In [ ]:
import json
import os
from pathlib import Path
import random
import pandas as pd

# 작업 파일/출력 경로
OUT_DIR = Path('finetune_output')
OUT_DIR.mkdir(exist_ok=True)
TRAIN_FILE = OUT_DIR / 'train.jsonl'
VAL_FILE = OUT_DIR / 'val.jsonl'

print('준비 완료')

In [ ]:
# 1) 데이터 준비: 현재 노트북 환경에서 wrong_df가 있으면 사용, 없으면 sample 데이터에서 생성
try:
    # wrong_df는 기존 노트북에서 생성된 오답 샘플 DataFrame (id,text,true_label,predicted_label,reason 등)
    source_df = wrong_df.copy()
    print('Using existing wrong_df with', len(source_df), 'rows')
except NameError:
    try:
        source_df = df.copy()
        print('wrong_df not found — falling back to df with', len(source_df), 'rows')
    except NameError:
        # 최소 예시 생성
        source_df = pd.DataFrame([
            {'id':'R02','category':'RIGHT','text':'사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.'},
            {'id':'P02','category':'PROC','text':'신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.'},
            {'id':'D01','category':'DEF','text':'이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.'},
        ])
        print('No existing DF; created minimal sample with', len(source_df), 'rows')

# Normalize columns: prefer true_label, else category
if 'true_label' in source_df.columns:
    source_df['label'] = source_df['true_label']
elif 'category' in source_df.columns:
    source_df['label'] = source_df['category']
else:
    source_df['label'] = 'ETC'

# For training, use label-only JSON completion to reduce PARSE_FAILs
def make_example(row):
    prompt = (
        '다음 법률 문장을 6개 코드 중 하나로 분류하라. 응답은 JSON으로 label만 포함하라.\n'
        f"문장: {row.get('text')}\n"
        "응답 예: {\"label\":\"RIGHT\"}"
    )
    completion = json.dumps({'label': row.get('label')}, ensure_ascii=False)
    return {'prompt': prompt, 'completion': completion}

examples = [make_example(r) for r in source_df.to_dict('records')]
random.shuffle(examples)
n = len(examples)
n_train = max(1, int(n * 0.8))
train_ex = examples[:n_train]
val_ex = examples[n_train:] or examples[:max(1, n - n_train)]

# Write JSONL files (prompt/completion)
with open(TRAIN_FILE, 'w', encoding='utf-8') as f:
    for ex in train_ex:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')
with open(VAL_FILE, 'w', encoding='utf-8') as f:
    for ex in val_ex:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

print('Wrote', len(train_ex), 'train and', len(val_ex), 'val examples to', OUT_DIR)

In [ ]:
# 오답 데이터를 기준으로 one few shot 데이터셋을 재구성
import json
data = []
with open(r'finetune_output\train.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        print(json.loads(line))

In [ ]:
data

generated_data.py

In [ ]:
import json
import random
from pathlib import Path

LABEL_DESC = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항',
}

TEMPLATES = {
    'DEF': [
        '이 규정에서 사용되는 용어의 정의는 다음과 같다: {term}은 {definition}을 말한다.',
        '{term}의 목적은 {purpose}에 있으며, 적용범위는 {scope}로 한다.',
    ],
    'RIGHT': [
        '사용자는 {action}할 권리를 가지며, 다음과 같은 의무를 진다: {duty}.',
        '사업자는 {prohibition}을 금지하며, 위반 시 {penalty}의 책임을 진다.',
    ],
    'PROC': [
        '신청은 {who}에게 제출하며, 심사는 {period} 이내에 완료한다.',
        '불복 절차는 {procedure}에 따라 진행되며, 이의신청은 {deadline}까지 가능하다.',
    ],
    'ORG': [
        '위원회는 {members}로 구성되며, 위원회의 권한은 {authority}로 한다.',
        '{org}를 설치하여 {task}를 수행한다.',
    ],
    'CRIT': [
        '응시자는 {qualification}을 갖추어야 하며, 자격 유효기간은 {period}이다.',
        '선정 기준은 {criteria}이며, 최소 점수는 {score} 이상으로 한다.',
    ],
    'ETC': [
        '이 규정은 {effective_date}부터 시행한다.',
        '경과조치는 {transition}에 따른다.',
    ],
}


def random_phrase(key):
    if key == 'term':
        return random.choice(['계약', '서비스', '회원'])
    if key == 'definition':
        return random.choice(['특정한 의미', '시스템 사용을 위한 조건', '이 문서에서 정의한 바'])
    if key == 'purpose':
        return random.choice(['공정한 운영', '안전한 서비스 제공', '정보 보호'])
    if key == 'scope':
        return random.choice(['국내 전역', '회원에게만', '모든 이용자에게'])
    if key == 'action':
        return random.choice(['정보 열람', '서비스 이용', '데이터 수정'])
    if key == 'duty':
        return random.choice(['정확한 정보 제공', '비밀번호 관리', '법령 준수'])
    if key == 'prohibition':
        return random.choice(['무단 복제', '부정 이용', '스팸 발송'])
    if key == 'penalty':
        return random.choice(['손해배상', '서비스 중지', '과태료'])
    if key == 'who':
        return random.choice(['관리자', '담당부서', '접수처'])
    if key == 'period':
        return random.choice(['14일', '30일', '60일'])
    if key == 'procedure':
        return random.choice(['서면 제출', '온라인 접수', '심사위원회 회부'])
    if key == 'deadline':
        return random.choice(['7일', '14일', '30일'])
    if key == 'members':
        return random.choice(['5인 이상', '위원장 포함 7인', '관련 전문가로 구성'])
    if key == 'authority':
        return random.choice(['결정권', '조정권', '심사권'])
    if key == 'org':
        return random.choice(['조정위원회', '전담부서'])
    if key == 'task':
        return random.choice(['심사', '집행', '감독'])
    if key == 'qualification':
        return random.choice(['학위 취득자', '관련 경력 3년 이상', '자격증 보유자'])
    if key == 'criteria':
        return random.choice(['성적', '경력', '면접 점수'])
    if key == 'score':
        return random.choice(['70', '80', '85'])
    if key == 'effective_date':
        return random.choice(['2026-01-01', '2026-06-01', '2026-12-01'])
    if key == 'transition':
        return random.choice(['종전 규정 준수', '유예 기간 6개월'])
    return '...'


def synthesize_clause(label):
    tpl = random.choice(TEMPLATES[label])
    # replace placeholders
    while '{' in tpl:
        start = tpl.find('{')
        end = tpl.find('}', start)
        key = tpl[start+1:end]
        tpl = tpl[:start] + random_phrase(key) + tpl[end+1:]
    return tpl


def make_prompt(clause):
    labels = ', '.join([f"{k}({v})" for k, v in LABEL_DESC.items()])
    prompt = (
        f"다음 조항을 다음 라벨 중 하나로 분류하시오. 가능한 라벨: {labels}\n"
        f"조항: {clause}\n라벨:"
    )
    return prompt


def generate(out_dir='./finetune_data', train_size=2000, val_size=400, seed=42):
    random.seed(seed)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    def write_file(path, n):
        with open(path, 'w', encoding='utf-8') as f:
            for _ in range(n):
                label = random.choice(list(LABEL_DESC.keys()))
                clause = synthesize_clause(label)
                prompt = make_prompt(clause)
                item = {'prompt': prompt, 'response': label}
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

    write_file(out / 'train.jsonl', train_size)
    write_file(out / 'validation.jsonl', val_size)
    print(f'Wrote train/validation to {out.resolve()}')


if __name__ == '__main__':
    generate()


# [4교시] ~ [5교시]
    - runpod 실습

##### 소규모 모델 PEFT(LoRA)튜닝
- 가상의 문장을 생성
- train.jsonl, validation.jsonl 생성
- Transformers + PEFT(LoRA) fineTurning

In [ ]:
import torch
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
train_file = r'finetune_data\train.jsonl'
valid_file = r'finetune_data\validation.jsonl'
output_dir = 'peft_output'
num_train_epochs = 3
per_device_train_batch_size = 4
learning_rate = 2e-4
# gpu
if torch.cuda.is_available():    
    use_bf16 = True
    use_fp16= False    
# cpu
else:
    use_bf16 = False
    use_fp16= True

lora_r = 8
lora_alpha = 16
lora_dropout = 0.05
use_4bit = False   
max_leng = 512

LABEL_DESC = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항',
}
print('Config set:', model_name)

In [ ]:
# Causal lM : 기본학습 다음토큰을 예측
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def build_inputs_and_labels(batch, tokenizer, max_length=512):
    '''prompt부분은 loss에서제외(-100) 정답 라벨 토큰만 학습'''
    inputs = []; labels = []
    for p, r in zip(batch['prompt'], batch['response']):
        full = p+' '+ r
        tokenized_full = tokenizer(full, truncation=True, max_length=max_length)
        tokenized_prompt = tokenizer(p, truncation=True, max_length=max_length)
        input_ids =  tokenized_full['input_ids']
        label_ids = input_ids.copy()

        prompt_len = len(tokenized_prompt['input_ids'])
        for i in range(min(prompt_len, len(label_ids))):
            label_ids[i]    = -100         
        inputs.append(input_ids)
        labels.append(label_ids)
    return {'input_ids' : inputs, 'labels':labels}

In [ ]:
# 토크나이져, 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
load_kwargs = {'trust_remote_code':True}
if use_4bit:
    load_kwargs.update({'load_in_4bit':True,'device_map':'auto'})
model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
if use_4bit:
    model = prepare_model_for_kbit_training(model)
print('model load')

##### LoRA 구성 및 적용
- 모델내부 선형층 이름을 스캔해 LoRA를 걸 target_modules를 자동으로 찾기
- attention/MLP projection에 LoRA를 적용?
    - 출력헤드 전체를 모두 fine tune하면 비용증가
    - 출력에 영향이 큰 선형층을 효율적으로 조정
- LoRA는 기존 가중치 W를 직접 바꾸지 않고 행렬 업데이트를통해서 조금 수정
- 학습 파라메터가 줄어서 cpu나 저성능의 gpu에서 가능

In [ ]:
import torch.nn as nn
def find_lora_target_modules(model):
    linear_leaf_names = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            linear_leaf_names.add(name.split('.')[-1])

    preferred = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    selected = [ m for m in preferred if m in linear_leaf_names]
    if not selected:
        fallback_keywords = ('proj','wq','wk','wv','wo','fc')
        selected = sorted([
            n for n in linear_leaf_names
            if any( k in  n for k in fallback_keywords) and n not in {'lm_head'}
        ])
    if not selected:
        raise ValueError(
            f'LoRA Target Module를 자동 탐색하지 못했습니다. 발견된 leaner leaf name : {linear_leaf_names}'
        )
    return selected, sorted(linear_leaf_names)

target_modules, linear_names = find_lora_target_modules(model)
print('Detected linear left names(samples) :',linear_names[:30])
print('Selected target_modules for LoRA', target_modules)

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=target_modules,
    lora_dropout=lora_dropout,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

#### 데이터셋 로드 및 전처리

In [ ]:
data_files = {'train':train_file, 'validation':valid_file}
dataset = load_dataset('json',data_files=data_files)

def preprocess_batch(batch):
    return build_inputs_and_labels(batch, tokenizer,max_length=max_leng)

tokenized_train = dataset['train'].map(
    lambda x : preprocess_batch(x), batched=True,remove_columns=dataset['train'].column_names
)
tokenized_eval = dataset['validation'].map(
    lambda x : preprocess_batch(x), batched=True,remove_columns=dataset['validation'].column_names
)

def collate_with_padding(features):
    # 가변 길이 샘플을 동적 패딩하고 labels는 -100으로 패딩
    input_ids = [f['input_ids'] for f in features]
    labels = [f['labels'] for f in features]

    padded = tokenizer.pad({'input_ids': input_ids}, padding=True, return_tensors='pt')
    max_len = padded['input_ids'].shape[1]

    padded_labels = []
    for lb in labels:
        padded_labels.append(lb + [-100] * (max_len - len(lb)))

    padded['labels'] = torch.tensor(padded_labels, dtype=torch.long)
    return padded


print('Datasets prepared:', len(tokenized_train), len(tokenized_eval))

##### Trainer 구성 및 학습시작

In [ ]:
from transformers import TrainingArguments, Trainer
base_kwargs = dict(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    bf16=use_bf16,
    fp16=use_fp16,
    save_total_limit=3,
    remove_unused_columns=False,
    logging_steps=10,
)

try:
    training_args =  TrainingArguments(**{**base_kwargs, 'evaluation_strategy':'epoch'})
except TypeError:
    training_args =  TrainingArguments(**base_kwargs)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator = collate_with_padding
)
train_result = trainer.train()
print(f"loss : {getattr(train_result,'training_loss', None)}")

# [6교시]

## runpod

### Qwen2.5_LoRA_Tune

In [ ]:
%pip install --upgrade pip
#%pip uninstall transformers -y
#%pip uninstall torch torchvision torchaudio -y
#%pip cache purge
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install transformers==4.52.4
%pip install accelerate peft bitsandbytes datasets sentencepiece tokenizers safetensors trl
from transformers import AutoModelForCausalLM, AutoTokenizer

In [1]:
import torch
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
train_file = r'./finetune_data/train.jsonl'
valid_file = r'./finetune_data/validation.jsonl'
output_dir = 'peft_output'
num_train_epochs = 3
per_device_train_batch_size = 4
learning_rate = 2e-4
# gpu
if torch.cuda.is_available():    
    use_bf16 = True
    use_fp16= False    
# cpu
else:
    use_bf16 = False
    use_fp16= True

lora_r = 8
lora_alpha = 16
lora_dropout = 0.05
use_4bit = False   
max_leng = 512

LABEL_DESC = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항',
}
print('Config set:', model_name)

Config set: Qwen/Qwen2.5-0.5B-Instruct


In [2]:
# Causal lM : 기본학습 다음토큰을 예측
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def build_inputs_and_labels(batch, tokenizer, max_length=512):
    '''prompt부분은 loss에서제외(-100) 정답 라벨 토큰만 학습'''
    inputs = []; labels = []
    for p, r in zip(batch['prompt'], batch['response']):
        full = p+' '+ r
        tokenized_full = tokenizer(full, truncation=True, max_length=max_length)
        tokenized_prompt = tokenizer(p, truncation=True, max_length=max_length)
        input_ids =  tokenized_full['input_ids']
        label_ids = input_ids.copy()

        prompt_len = len(tokenized_prompt['input_ids'])
        for i in range(min(prompt_len, len(label_ids))):
            label_ids[i]    = -100         
        inputs.append(input_ids)
        labels.append(label_ids)
    return {'input_ids' : inputs, 'labels':labels}

In [3]:
# 토크나이져, 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
load_kwargs = {'trust_remote_code':True}
if use_4bit:
    load_kwargs.update({'load_in_4bit':True,'device_map':'auto'})
model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
if use_4bit:
    model = prepare_model_for_kbit_training(model)
print('model load')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

model load


In [4]:
import torch.nn as nn
def find_lora_target_modules(model):
    linear_leaf_names = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            linear_leaf_names.add(name.split('.')[-1])

    preferred = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    selected = [ m for m in preferred if m in linear_leaf_names]
    if not selected:
        fallback_keywords = ('proj','wq','wk','wv','wo','fc')
        selected = sorted([
            n for n in linear_leaf_names
            if any( k in  n for k in fallback_keywords) and n not in {'lm_head'}
        ])
    if not selected:
        raise ValueError(
            f'LoRA Target Module를 자동 탐색하지 못했습니다. 발견된 leaner leaf name : {linear_leaf_names}'
        )
    return selected, sorted(linear_leaf_names)

target_modules, linear_names = find_lora_target_modules(model)
print('Detected linear left names(samples) :',linear_names[:30])
print('Selected target_modules for LoRA', target_modules)

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=target_modules,
    lora_dropout=lora_dropout,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Detected linear left names(samples) : ['down_proj', 'gate_proj', 'k_proj', 'lm_head', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
Selected target_modules for LoRA ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [5]:
data_files = {'train':train_file, 'validation':valid_file}
dataset = load_dataset('json',data_files=data_files)

def preprocess_batch(batch):
    return build_inputs_and_labels(batch, tokenizer,max_length=max_leng)

tokenized_train = dataset['train'].map(
    lambda x : preprocess_batch(x), batched=True,remove_columns=dataset['train'].column_names
)
tokenized_eval = dataset['validation'].map(
    lambda x : preprocess_batch(x), batched=True,remove_columns=dataset['validation'].column_names
)

def collate_with_padding(features):
    # 가변 길이 샘플을 동적 패딩하고 labels는 -100으로 패딩
    input_ids = [f['input_ids'] for f in features]
    labels = [f['labels'] for f in features]

    padded = tokenizer.pad({'input_ids': input_ids}, padding=True, return_tensors='pt')
    max_len = padded['input_ids'].shape[1]

    padded_labels = []
    for lb in labels:
        padded_labels.append(lb + [-100] * (max_len - len(lb)))

    padded['labels'] = torch.tensor(padded_labels, dtype=torch.long)
    return padded


print('Datasets prepared:', len(tokenized_train), len(tokenized_eval))

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Datasets prepared: 2000 400


In [6]:
from transformers import TrainingArguments, Trainer
base_kwargs = dict(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    bf16=use_bf16,
    fp16=use_fp16,
    save_total_limit=3,
    remove_unused_columns=False,
    logging_steps=10,
)

try:
    training_args =  TrainingArguments(**{**base_kwargs, 'evaluation_strategy':'epoch'})
except TypeError:
    training_args =  TrainingArguments(**base_kwargs)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator = collate_with_padding
)
train_result = trainer.train()
print(f"loss : {getattr(train_result,'training_loss', None)}")

c:\miniconda\envs\edu_env\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
# 학습완료후 PEFT 어뎁터 저장
model.save_pretrained(output_dir)
print('saved PEFT adapters : ',output_dir)

In [ ]:
from peft import PeftModel
reload_kwargs = {'trust_remote_code':True}
if use_4bit:
    reload_kwargs.update({'load_in_4bit' : True, 'device_map':'auto'})
else:
    reload_kwargs.update({
        'torch_dtype':torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    })
base_model_for_eval = AutoModelForCausalLM.from_pretrained(model_name, **reload_kwargs)    
model = PeftModel.from_pretrained(base_model_for_eval,output_dir)
if torch.cuda.is_available() and not use_4bit:
    model.to('cuda')
model.eval()
print('reload PEFT adapter from:',output_dir)
print('Eval model dtype:',next(model.parameters()).dtype)


In [ ]:
import random

label_list = list(LABEL_DESC.keys())


def predict_label_from_prompt(prompt_text, max_new_tokens=6):
    model.eval()
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    upper = generated.upper()
    for lb in label_list:
        if lb in upper:
            return lb, generated

    first = generated.split()[0].strip(".,:;()[]{}\"'").upper() if generated else ''
    if first in label_list:
        return first, generated
    return 'ETC', generated


val_ds = load_dataset('json', data_files={'validation': valid_file})['validation']
num_eval = min(100, len(val_ds))
indices = random.sample(range(len(val_ds)), k=num_eval)

correct = 0
samples = []
for idx in indices:
    item = val_ds[idx]
    pred, raw = predict_label_from_prompt(item['prompt'])
    gt = item['response']
    correct += int(pred == gt)
    if len(samples) < 5:
        samples.append((gt, pred, raw))

acc = correct / num_eval if num_eval else 0.0
print(f'Validation sample accuracy ({num_eval}개): {acc:.4f}')
print('\n예시 5개 (정답, 예측, 생성원문):')
for row in samples:
    print(row)

In [ ]:
# 단건 추론
test_clause = '신청인은 접수일로부터 14일 이내에 이의신청서를 제출해야 하며, 담당 기관은 30일 내에 심사 결과를 통지한다.'
label_candidates = ', '.join([f"{k}({v})" for k, v in LABEL_DESC.items()])
test_prompt = f"다음 조항을 다음 라벨 중 하나로 분류하시오. 가능한 라벨: {label_candidates}\n조항: {test_clause}\n라벨:"

pred, raw = predict_label_from_prompt(test_prompt)
print('입력 조항:', test_clause)
print('예측 라벨:', pred)
print('생성 원문:', raw)

# [7교시]

## QWen2.5_load

In [ ]:
import zipfile

In [ ]:
output_dir = 'peft_output'
with zipfile.ZipFile('peft_output.zip','r') as f:
    f.extractall(output_dir)

In [ ]:
%pip install --upgrade pip
#%pip uninstall transformers -y
#%pip uninstall torch torchvision torchaudio -y
#%pip cache purge
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install transformers==4.52.4
%pip install accelerate peft bitsandbytes datasets sentencepiece tokenizers safetensors trl
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
output_dir = 'peft_output/peft_output'
use_4bit = False
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'

from peft import PeftModel
reload_kwargs = {'trust_remote_code':True}
if use_4bit:
    reload_kwargs.update({'load_in_4bit' : True, 'device_map':'auto'})
else:
    reload_kwargs.update({
        'dtype':torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    })
base_model_for_eval = AutoModelForCausalLM.from_pretrained(model_name, **reload_kwargs)    
model = PeftModel.from_pretrained(base_model_for_eval,output_dir)
if torch.cuda.is_available() and not use_4bit:
    model.to('cuda')
model.eval()
print('reload PEFT adapter from:',output_dir)
print('Eval model dtype:',next(model.parameters()).dtype)


In [ ]:
import random
LABEL_DESC = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항',
}

label_list = list(LABEL_DESC.keys())

valid_file = r'./validation.jsonl'

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

def predict_label_from_prompt(prompt_text, max_new_tokens=6):
    model.eval()
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    upper = generated.upper()
    for lb in label_list:
        if lb in upper:
            return lb, generated

    first = generated.split()[0].strip(".,:;()[]{}\"'").upper() if generated else ''
    if first in label_list:
        return first, generated
    return 'ETC', generated


val_ds = load_dataset('json', data_files={'validation': valid_file})['validation']
num_eval = min(100, len(val_ds))
indices = random.sample(range(len(val_ds)), k=num_eval)

correct = 0
samples = []
for idx in indices:
    item = val_ds[idx]
    pred, raw = predict_label_from_prompt(item['prompt'])
    gt = item['response']
    correct += int(pred == gt)
    if len(samples) < 5:
        samples.append((gt, pred, raw))

acc = correct / num_eval if num_eval else 0.0
print(f'Validation sample accuracy ({num_eval}개): {acc:.4f}')
print('\n예시 5개 (정답, 예측, 생성원문):')
for row in samples:
    print(row)